In [8]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scripts import training
from torch.utils.data import TensorDataset, DataLoader

In [2]:
#reminder of variables/ease for pulling csv files later
train1 = "10132000"
train2 = "10136600"
train3 = "10137000"
test = "10136500"
start = "2016-01-01"
end="2025-12-31"

#X represents input to the model (aka "parameters"), Y is the streamflow (aka "output")
#pull dataframes, keeping separate for now for organization/debugging
X_site_1 = pd.read_csv(f'Data Files/{train1}_combined.csv')
X_site_2 = pd.read_csv(f'Data Files/{train2}_combined.csv')
X_site_3 = pd.read_csv(f'Data Files/{train3}_combined.csv')
# will combine into one dataset when they are tensors (after sequencing)  

#training streamflows
Y_site_1 = pd.read_csv(f'Data Files/Streamflow/{train1}_discharge.csv')
Y_site_2 = pd.read_csv(f'Data Files/Streamflow/{train2}_discharge.csv')
Y_site_3 = pd.read_csv(f'Data Files/Streamflow/{train3}_discharge.csv')

for my memory, doing 2016-2022 then 2023-2025 has 30% validation 
               doing 2016-2023 then 2024-2025 has 20% validation 

In [3]:
#prep for splitting by year
X_site_1['year'] = pd.to_datetime(X_site_1['Date']).dt.year
X_site_2['year'] = pd.to_datetime(X_site_2['Date']).dt.year
X_site_3['year'] = pd.to_datetime(X_site_3['Date']).dt.year

#80/20 split
training_start =2016
training_end =2023
validate_start =2024
validate_end =2025

#develop training and validation input sets for each site
(X_training_1, X_training_2,X_training_3,
 X_validate_1,X_validate_2, X_validate_3) = training.train_validation_split(X_site_1,
                                                                            X_site_2,
                                                                            X_site_3,
                                                                            training_end,
                                                                            validate_start,
                                                                            validate_end,
                                                                            'input')


#develop training and validation output sets for each site
(Y_training_1, Y_training_2,Y_training_3,
 Y_validate_1,Y_validate_2, Y_validate_3) = training.train_validation_split(Y_site_1,
                                                                            Y_site_2,
                                                                            Y_site_3,
                                                                            training_end,
                                                                            validate_start,
                                                                            validate_end,
                                                                            "target")


Training Rows: Site 1 - 2922, Site 2 = 2922, Site 3 - 2922
Validation Rows: Site 1 - 731, Site 2 = 731, Site 3 - 731
Percent Validation: 20%
Training Rows: Site 1 - 2922, Site 2 = 2922, Site 3 - 2922
Validation Rows: Site 1 - 731, Site 2 = 731, Site 3 - 731
Percent Validation: 20%


convert to tensors for model 

In [4]:
# #training sets for all sites 
# X_train = torch.cat([
#     torch.tensor(X_training_1.values, dtype=torch.float32),
#     torch.tensor(X_training_2.values, dtype=torch.float32),
#     torch.tensor(X_training_3.values, dtype=torch.float32)
# ], dim=0)

# Y_train = torch.cat([
#     torch.tensor(Y_training_1.values, dtype=torch.float32),
#     torch.tensor(Y_training_2.values, dtype=torch.float32),
#     torch.tensor(Y_training_3.values, dtype=torch.float32)
# ], dim=0)

# X_valid = torch.cat([
#     torch.tensor(X_validate_1.values, dtype=torch.float32),
#     torch.tensor(X_validate_2.values, dtype=torch.float32),
#     torch.tensor(X_validate_3.values, dtype=torch.float32)
# ], dim=0)

# Y_valid = torch.cat([
#     torch.tensor(Y_validate_1.values, dtype=torch.float32),
#     torch.tensor(Y_validate_2.values, dtype=torch.float32),
#     torch.tensor(Y_validate_3.values, dtype=torch.float32)
# ], dim=0)

Now build the mask, so the nan values aren't trained on. Then the sequences are created in 30-day lengths 

In [5]:
X_t1, mask_t1 = training.masking(X_training_1)
X_t2, mask_t2 = training.masking(X_training_2)
X_t3, mask_t3 = training.masking(X_training_3)

X_v1, mask_v1 = training.masking(X_validate_1)
X_v2, mask_v2 = training.masking(X_validate_2)
X_v3, mask_v3 = training.masking(X_validate_3)

Now lets do the normalization.

In [6]:
(X_t1_norm, X_t2_norm,X_t3_norm,
X_v1_norm,X_v2_norm,X_v3_norm) = training.minmax(X_t1, X_t2,X_t3,
                                                 X_v1,X_v2,X_v3,'x')

(Y_t1_norm, Y_t2_norm,Y_t3_norm,
Y_v1_norm,Y_v2_norm,Y_v3_norm) = training.minmax(Y_training_1, Y_training_2,Y_training_3,
                                                 Y_validate_1,Y_validate_2, Y_validate_3,'y')

building sequences

In [7]:
X_train, M_train, Y_train = training.build_full_sets([X_t1_norm, X_t2_norm,X_t3_norm],[mask_t1,mask_t2,mask_t3],[Y_t1_norm, Y_t2_norm,Y_t3_norm])

X_valid, M_valid, Y_valid = training.build_full_sets([X_v1_norm,X_v2_norm,X_v3_norm],[mask_v1,mask_v2,mask_v3],[Y_v1_norm,Y_v2_norm,Y_v3_norm])

/Users/lizzieburgon/Hydroinformatics/Homework3/scripts/training.py:91: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729647065806/work/torch/csrc/utils/tensor_new.cpp:281.)
  xset = torch.tensor(xset, dtype=torch.float32)


Now lets load the data for the LSTM

In [9]:
train_loaded = DataLoader(
    TensorDataset(X_train, M_train, Y_train),
    batch_size=64,
    shuffle=True
)

val_loaded = DataLoader(
    TensorDataset(X_valid, M_valid, Y_valid),
    batch_size=64,
    shuffle=False
)